#  Orcaopta Notebook Studio

Welcome to the Orcaopta Cloud Brain.

##  Quick Actions
- GPU Health
- Kubernetes Audit
- Ceph Health
- Start Supervisor
- Cloud Graph Viewer

##  Workflows
- Training (`train.ipynb`)
- Evaluation (`evaluate.ipynb`)
- Tuning (`tune.ipynb`)
- RL Training (`rl_train.ipynb`)
- RL Evaluation (`rl_evaluate.ipynb`)
- MLflow Experiments (`mlflow.ipynb`)

##  Environment
- Auto GPU/CPU detection  
- JupyterLab power-up mode  
- Integrated MCP tools  
- Supervisor control  


In [ ]:
from kitaru.mcp import MCPClient
from orcaopta.utils.device import device

client = MCPClient()

print(f"Orcaopta Studio — device: {device}")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

model_selector = widgets.Dropdown(
    options=["anomaly", "forecast", "resource_opt", "autoscale", "ppo_agent"],
    description="Model:",
)

dataset_selector = widgets.Dropdown(
    options=["cluster_metrics", "node_stats", "gpu_profile"],
    description="Dataset:",
)

env_selector = widgets.ToggleButtons(
    options=["CPU", "GPU"],
    description="Device:",
)

display(model_selector, dataset_selector, env_selector)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

metrics_panel = widgets.Output()
logs_panel = widgets.Output()
cluster_panel = widgets.Output()

accordion = widgets.Accordion(children=[metrics_panel, logs_panel, cluster_panel])
accordion.set_title(0, " Metrics")
accordion.set_title(1, " Logs")
accordion.set_title(2, " Cluster Status")

display(accordion)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

def on_gpu_click(b):
    result = client.call("gpu_health")
    with metrics_panel:
        metrics_panel.clear_output()
        print("GPU Health:")
        print(result)

def on_k8s_click(b):
    result = client.call("tool_kubernetes_audit")
    with logs_panel:
        logs_panel.clear_output()
        print("Kubernetes Audit:")
        print(result)

def on_ceph_click(b):
    result = client.call("tool_ceph_health")
    with logs_panel:
        print("Ceph Health:")
        print(result)

def on_cloud_graph_click(b):
    result = client.call("tool_cloud_graph")
    with cluster_panel:
        cluster_panel.clear_output()
        print("Cloud Graph:")
        print(result)

def on_supervisor_click(b):
    result = client.call("tool_start_supervisor")
    with logs_panel:
        print("Supervisor:")
        print(result)

gpu_button = widgets.Button(description="Check GPU Health", button_style="info")
k8s_button = widgets.Button(description="Run K8s Audit", button_style="warning")
ceph_button = widgets.Button(description="Check Ceph Health", button_style="warning")
cloud_graph_button = widgets.Button(description="Show Cloud Graph", button_style="primary")
supervisor_button = widgets.Button(description="Start Supervisor", button_style="danger")

gpu_button.on_click(on_gpu_click)
k8s_button.on_click(on_k8s_click)
ceph_button.on_click(on_ceph_click)
cloud_graph_button.on_click(on_cloud_graph_click)
supervisor_button.on_click(on_supervisor_click)

buttons_box = widgets.HBox([
    gpu_button,
    k8s_button,
    ceph_button,
    cloud_graph_button,
    supervisor_button,
])

display(buttons_box)


In [ ]:
import time
import threading

cluster_output = widgets.Output()

def update_cluster():
    while True:
        try:
            result = client.call("cluster_health")
            with cluster_output:
                cluster_output.clear_output()
                print("Cluster Health:")
                print(result)
        except Exception as e:
            with cluster_output:
                print(f"Error updating cluster health: {e}")
        time.sleep(5)

threading.Thread(target=update_cluster, daemon=True).start()
display(cluster_output)


In [ ]:
ml_output = widgets.Output()
rl_output = widgets.Output()

def on_ml_signals_click(b):
    result = client.call("tool_ml_signals")
    with ml_output:
        ml_output.clear_output()
        print("ML Signals:")
        print(result)

def on_rl_signals_click(b):
    result = client.call("tool_rl_signals")
    with rl_output:
        rl_output.clear_output()
        print("RL Signals:")
        print(result)

ml_button = widgets.Button(description="Get ML Signals", button_style="info")
rl_button = widgets.Button(description="Get RL Signals", button_style="info")

ml_button.on_click(on_ml_signals_click)
rl_button.on_click(on_rl_signals_click)

display(widgets.HBox([ml_button, rl_button]))
display(ml_output, rl_output)


In [ ]:
from src.orcaopta.mcp.worker import MCPWorker

w = MCPWorker()

w.call("gpu_profiler")
w.call("ml_signals")
w.call("system_mode")
